# Metadata-only biomass baseline with AutoGluon

Estimate how much predictive signal is present in five non-image fields: Sampling_Date, State, Species, Pre_GSHH_NDVI, and Height_Ave_cm. The notebook aggregates the long-format competition labels to one row per image, trains one AutoGluon regression predictor per biomass target, and evaluates out-of-fold predictions with the competition weighted R2.

AutoGluon uses five-fold bagging. The image ID is used only to align rows and is never provided as a model feature. The noisy image ID excluded by the current baseline is excluded here as well. No image pixels, pretrained image features, or biomass postprocessing are used.

If AutoGluon Tabular is not installed in the notebook environment, install it in a separate cell with %pip install autogluon.tabular when package installation is available.

In [1]:
from pathlib import Path
import gc
import tempfile
import warnings

import numpy as np
import pandas as pd
from autogluon.tabular import TabularPredictor

warnings.filterwarnings("ignore")

TARGET_COLS = [
    "Dry_Green_g",
    "Dry_Dead_g",
    "Dry_Clover_g",
    "GDM_g",
    "Dry_Total_g",
]
TARGET_WEIGHTS = np.array([0.1, 0.1, 0.1, 0.2, 0.5], dtype=float)
SOURCE_FEATURES = [
    "Sampling_Date",
    "State",
    "Species",
    "Pre_GSHH_NDVI",
    "Height_Ave_cm",
]
EXCLUDED_IMAGE_ID = "ID230058600"
NUM_FOLDS = 5
RANDOM_SEED = 42

def find_train_csv():
    candidates = [
        Path("data/train.csv"),
        Path("../data/train.csv"),
        Path("/kaggle/input/csiro-biomass/train.csv"),
    ]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        candidates.extend(kaggle_input.glob("*/train.csv"))
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find train.csv. Place it under data/ or attach the competition data in Kaggle."
    )

def weighted_r2_score(y_true, y_pred):
    """Calculate the competition weighted R2 over all target columns."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    residual_sum = np.sum(TARGET_WEIGHTS * np.sum((y_true - y_pred) ** 2, axis=0))
    weighted_mean = np.average(np.mean(y_true, axis=0), weights=TARGET_WEIGHTS)
    total_sum = np.sum(TARGET_WEIGHTS * np.sum((y_true - weighted_mean) ** 2, axis=0))
    return 1.0 - residual_sum / (total_sum + 1e-8)

train_csv = find_train_csv()
print(f"Training file: {train_csv}")
print(f"AutoGluon: {__import__('autogluon').__version__}")

ModuleNotFoundError: No module named 'autogluon'

In [ ]:
# Read the long-format CSV and build one row per image.
raw_train = pd.read_csv(train_csv)
raw_train["image_id"] = raw_train["sample_id"].str.split("__", n=1).str[0]

missing_columns = sorted(set(["image_id", *SOURCE_FEATURES, "target_name", "target"]) - set(raw_train.columns))
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Confirm that each image has one consistent metadata record across its five targets.
metadata_counts = raw_train.groupby("image_id")[SOURCE_FEATURES].nunique(dropna=False)
if (metadata_counts > 1).any().any():
    raise ValueError("Metadata values are inconsistent across target rows for at least one image.")

image_features = (
    raw_train[["image_id", *SOURCE_FEATURES]]
    .drop_duplicates(subset="image_id")
    .set_index("image_id")
)
target_wide = (
    raw_train.pivot(index="image_id", columns="target_name", values="target")
    .reindex(columns=TARGET_COLS)
)
if target_wide.isna().any().any():
    raise ValueError("At least one image is missing a biomass target.")

# Align metadata and targets by image ID before creating the cross-validation table.
image_features = image_features.reindex(target_wide.index)
data = image_features.join(target_wide)
if EXCLUDED_IMAGE_ID in data.index:
    data = data.drop(index=EXCLUDED_IMAGE_ID)
    print(f"Excluded {EXCLUDED_IMAGE_ID} to match the current baseline.")
print(f"Images used for cross-validation: {len(data)}")

# Expand Sampling_Date into numeric calendar features. State and Species remain categorical.
sampling_date = pd.to_datetime(data["Sampling_Date"], errors="coerce")
model_features = pd.DataFrame(index=data.index)
model_features["Sampling_Year"] = sampling_date.dt.year
model_features["Sampling_Month"] = sampling_date.dt.month
model_features["Sampling_DayOfYear"] = sampling_date.dt.dayofyear
model_features["Sampling_Month_Sin"] = np.sin(2 * np.pi * sampling_date.dt.month / 12)
model_features["Sampling_Month_Cos"] = np.cos(2 * np.pi * sampling_date.dt.month / 12)
model_features["State"] = data["State"].astype("category")
model_features["Species"] = data["Species"].astype("category")
model_features["Pre_GSHH_NDVI"] = pd.to_numeric(data["Pre_GSHH_NDVI"], errors="coerce")
model_features["Height_Ave_cm"] = pd.to_numeric(data["Height_Ave_cm"], errors="coerce")
targets = data[TARGET_COLS].astype(float)

print(f"Model feature columns: {model_features.columns.tolist()}")
print(f"Target columns: {TARGET_COLS}")
display(model_features.head())

In [ ]:
# Fit five-fold AutoGluon models and collect one out-of-fold prediction per image and target.
oof_predictions = pd.DataFrame(index=model_features.index, columns=TARGET_COLS, dtype=float)
best_models = {}

for target in TARGET_COLS:
    print(f"Training AutoGluon for {target}...")
    train_data = model_features.copy()
    train_data[target] = targets[target].to_numpy()

    with tempfile.TemporaryDirectory(prefix=f"ag_metadata_{target}_") as model_path:
        predictor = TabularPredictor(
            label=target,
            problem_type="regression",
            eval_metric="r2",
            path=model_path,
            verbosity=2,
        )
        predictor.fit(
            train_data=train_data,
            presets="medium_quality",
            num_bag_folds=NUM_FOLDS,
            num_bag_sets=1,
            num_stack_levels=0,
            fit_weighted_ensemble=False,
        )
        best_model = predictor.model_best
        oof_series = predictor.predict_oof(
            model=best_model,
            train_data=train_data,
        )
        if len(oof_series) != len(train_data):
            raise ValueError(
                f"AutoGluon returned {len(oof_series)} OOF predictions for "
                f"{len(train_data)} rows in {target}."
            )
        oof_predictions[target] = np.asarray(oof_series, dtype=float)
        best_models[target] = best_model
        print(f"Best AutoGluon model for {target}: {best_model}")

    del predictor
    gc.collect()

# Clip negative biomass predictions, consistent with the baseline prediction pipeline.
oof_predictions = oof_predictions.clip(lower=0)

In [ ]:
# Report the single competition-style score from the combined out-of-fold predictions.
if oof_predictions.isna().any().any():
    raise ValueError("Some target rows do not have an out-of-fold prediction.")

cv_score = weighted_r2_score(
    targets[TARGET_COLS].to_numpy(),
    oof_predictions[TARGET_COLS].to_numpy(),
)
print(f"5-fold AutoGluon metadata-only CV weighted R2: {cv_score:.5f}")